# 37 — Operating Regimes Preserve Computational Admissibility

**Engineering question**

Where does a quantum photonic architecture actually work?

Notebook 31 established that engineering allocation should follow admissibility.  
Notebook 37 asks how those admitted resources become a stable operating regime:

```text
physics
→ resource generation
→ admissibility
→ connected admissible region
→ operating regime
→ fault-tolerant computation
```

Engineering statement:

> Resources become computationally useful only inside the largest connected region satisfying every active specification.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/37_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Operating window

The goal is not to maximize squeezing in isolation or minimize loss in isolation.

The engineering goal is to maintain the **largest connected admissible region**: the stable part of parameter space where active specifications remain jointly satisfied.

This figure uses an illustrative admissibility score:

```text
admissibility = squeezing benefit × low-loss benefit × stability envelope
```


In [ ]:
s = np.linspace(0, 1, 260)
l = np.linspace(0, 1, 260)
S, L = np.meshgrid(s, l)

squeezing_benefit = 1 / (1 + np.exp(-14 * (S - 0.34)))
loss_penalty = np.exp(-3.2 * L)
overdrive_penalty = np.exp(-8.0 * np.maximum(S - 0.83, 0) ** 2)
thermal_penalty = np.exp(-5.5 * np.maximum(S + 0.75 * L - 1.08, 0) ** 2)
score = squeezing_benefit * loss_penalty * overdrive_penalty * thermal_penalty
score = score / score.max()

fig, ax = plt.subplots(figsize=(8.8, 5.5))
im = ax.imshow(score, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0, vmax=1)

contours = ax.contour(S, L, score, levels=[0.35, 0.55, 0.72], colors="black", linewidths=[1.4, 1.9, 2.7])
ax.clabel(contours, inline=True, fontsize=8)

target_x, target_y = 0.58, 0.22
ax.scatter([target_x], [target_y], s=220, zorder=5)
ax.annotate("design target", xy=(target_x, target_y), xytext=(0.72, 0.34),
            arrowprops=dict(arrowstyle="->", linewidth=1.8), fontsize=10)

ax.text(0.47, 0.49, "largest connected\nadmissible region", ha="center", fontsize=11)
ax.text(0.15, 0.82, "under-driven:\ninsufficient squeezing", ha="center", fontsize=10)
ax.text(0.83, 0.82, "unstable:\nover-driven", ha="center", fontsize=10)
ax.text(0.84, 0.11, "useful but\nloss-limited", ha="center", fontsize=10)

ax.set_title("Operating Window: Preserve the Largest Connected Admissible Region", fontsize=15)
ax.set_xlabel("Squeezing / nonlinear interaction strength")
ax.set_ylabel("Optical loss")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("admissibility score")

savefig(fig, "37_01_operating_window.png")
plt.show()


## 2. Operating regimes are constraint intersections

A usable operating regime is the common intersection of active specifications.

Each single specification may be satisfiable in isolation.

Useful computation exists only where every active specification overlaps.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.2))
ax.set_aspect("equal")
ax.axis("off")

constraint_names = [
    ("loss", (-0.20, 0.05), 0.50),
    ("graph", (0.20, 0.05), 0.50),
    ("timing", (0.00, 0.39), 0.50),
    ("detection", (-0.08, -0.29), 0.50),
    ("calibration", (0.32, -0.21), 0.50),
]

for name, center, radius in constraint_names:
    circ = Circle(center, radius, fill=True, alpha=0.12, linewidth=2.2)
    ax.add_patch(circ)
    outline = Circle(center, radius, fill=False, linewidth=2.2, alpha=0.9)
    ax.add_patch(outline)
    ax.text(center[0], center[1] + radius + 0.06, name, ha="center", fontsize=11)

center_region = Circle((0.05, 0.02), 0.17, fill=True, alpha=0.45)
ax.add_patch(center_region)
ax.text(0.05, 0.02, "connected\nadmissible\nregion", ha="center", va="center", fontsize=10)

ax.set_xlim(-0.88, 0.92)
ax.set_ylim(-0.88, 0.98)
ax.set_title("Operating Regimes Are Constraint Intersections", fontsize=16)
ax.text(0.02, -0.81, "useful computation exists only where every active specification overlaps", ha="center", fontsize=10)

savefig(fig, "37_02_constraint_intersection.png")
plt.show()


## 3. Active specifications by operating regime

A regime is governed only by specifications that are active in that region.

Some specifications are:

- **required**
- **helpful**
- **dormant**

This figure replaces the idea of a generic budget with an engineering dashboard.


In [ ]:
categories = ["physics", "engineering", "calibration", "feedback", "timing", "readout"]
required = np.array([0.55, 0.45, 0.35, 0.22, 0.25, 0.40])
helpful = np.array([0.25, 0.35, 0.30, 0.34, 0.30, 0.28])
dormant = 1 - required - helpful

x = np.arange(len(categories))
fig, ax = plt.subplots(figsize=(9.6, 5.2))
ax.bar(x, required, label="required")
ax.bar(x, helpful, bottom=required, label="helpful")
ax.bar(x, dormant, bottom=required + helpful, label="dormant")

ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("relative specification status")
ax.set_title("Active Specifications by Operating Regime", fontsize=16)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.25)

for xi, r, h, d in zip(x, required, helpful, dormant):
    ax.text(xi, r / 2, "✓", ha="center", va="center", fontsize=14)
    ax.text(xi, r + h / 2, "•", ha="center", va="center", fontsize=14)
    ax.text(xi, r + h + d / 2, "○", ha="center", va="center", fontsize=14)

budget = pd.DataFrame({
    "category": categories,
    "required": required,
    "helpful": helpful,
    "dormant": dormant,
})
budget.to_csv(CSV / "37_active_specifications.csv", index=False)
budget.to_json(JS / "37_active_specifications.json", orient="records", indent=2)

savefig(fig, "37_03_active_specifications.png")
plt.show()
budget


## 4. Design progression

A generated resource is not automatically useful.

The progression is:

```text
Generated resources
→ Admitted resources
→ Largest connected admissible region
→ Stable operating regime
→ Fault-tolerant computation
```

Each engineering stage specifies the next.


In [ ]:
labels = [
    "Generated\nresources",
    "Admitted\nresources",
    "Largest connected\nadmissible region",
    "Stable\noperating regime",
    "Fault-tolerant\ncomputation",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(12.5, 3.7))
ax.axis("off")

for i, label in enumerate(labels):
    rect = Rectangle((i - 0.50, -0.25), 1.00, 0.50, fill=False, linewidth=2.0)
    ax.add_patch(rect)
    ax.text(i, 0, label, ha="center", va="center", fontsize=9.3)
    if i < len(labels) - 1:
        ax.annotate("", xy=(i + 0.64, 0), xytext=(i + 0.50, 0), arrowprops=dict(arrowstyle="->", linewidth=2))

stage_notes = ["physics", "specification", "connected region", "stability"]
for i, note in enumerate(stage_notes):
    ax.text(i + 0.5, 0.42, note, ha="center", fontsize=9)

ax.text((len(labels) - 1) / 2, -0.56, "each engineering stage specifies the next", ha="center", fontsize=10.5)
ax.set_xlim(-0.74, len(labels) - 0.26)
ax.set_ylim(-0.75, 0.68)
ax.set_title("Design Progression: From Generated Resources to Fault-Tolerant Computation", fontsize=15)

savefig(fig, "37_04_design_progression.png")
plt.show()


## 5. Operating region between two failure modes

The operating region usually lies between failure modes.

Too little nonlinear interaction under-drives the architecture.  
Too much can over-drive the architecture through instability, heating, drift, or degraded control.

The design target is not a point optimum. It is a connected plateau.


In [ ]:
x = np.linspace(0, 1, 260)
y = np.linspace(0, 1, 260)
X, Y = np.meshgrid(x, y)

left_failure = 1 / (1 + np.exp(16 * (X - 0.25)))
right_failure = 1 / (1 + np.exp(-16 * (X - 0.78)))
loss_failure = 1 / (1 + np.exp(-10 * (Y - 0.70)))
center_plateau = np.exp(-((X - 0.52)**2 / 0.09 + (Y - 0.34)**2 / 0.18))
plateau = center_plateau * (1 - 0.55 * loss_failure)
plateau = plateau / plateau.max()

fig, ax = plt.subplots(figsize=(8.8, 5.5))
im = ax.imshow(plateau, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0, vmax=1)

contours = ax.contour(X, Y, plateau, levels=[0.35, 0.55, 0.75], colors="black", linewidths=[1.4, 1.9, 2.5])
ax.clabel(contours, inline=True, fontsize=8)

ax.text(0.16, 0.18, "under-driven", ha="center", fontsize=11)
ax.text(0.52, 0.46, "largest connected\nadmissible region", ha="center", fontsize=11)
ax.text(0.84, 0.18, "over-driven", ha="center", fontsize=11)
ax.text(0.78, 0.82, "loss-dominated", ha="center", fontsize=10)

ax.set_title("Operating Region Exists Between Two Failure Modes", fontsize=16)
ax.set_xlabel("nonlinear interaction / squeezing drive")
ax.set_ylabel("loss / instability pressure")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("admissibility")

savefig(fig, "37_05_failure_mode_contour.png")
plt.show()


## 6. Engineering summary

Notebook 37 introduces a new row that prepares Notebook 43:

```text
Robustness asks how much drift is tolerated before the connected region fails.
```


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Stage": "Physics",
            "Question": "What can exist?",
            "Maintained quantity": "Optical resources",
        },
        {
            "Stage": "Admissibility",
            "Question": "What survives?",
            "Maintained quantity": "Admitted modes",
        },
        {
            "Stage": "Allocation",
            "Question": "What deserves effort?",
            "Maintained quantity": "Bottleneck specifications",
        },
        {
            "Stage": "Operating regime",
            "Question": "Where do specifications overlap?",
            "Maintained quantity": "Connected admissible region",
        },
        {
            "Stage": "Robustness",
            "Question": "How much drift is tolerated?",
            "Maintained quantity": "Operating margin",
        },
        {
            "Stage": "Computation",
            "Question": "Which logical operations remain?",
            "Maintained quantity": "Fault-tolerant execution",
        },
    ]
)

fig, ax = plt.subplots(figsize=(13.5, 4.0))
ax.axis("off")

table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9.3)
table.scale(1.08, 1.75)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Operating Regimes Preserve Admissibility", fontsize=16, pad=20)
ax.text(0.5, -0.10, "Engineering preserves the connected admissible region that supports computation.",
        ha="center", transform=ax.transAxes, fontsize=10.5)

summary.to_csv(CSV / "37_summary.csv", index=False)
summary.to_json(JS / "37_summary.json", orient="records", indent=2)

savefig(fig, "37_06_summary_table.png")
plt.show()
summary


## 7. Architecture decision matrix

The operating-regime question is also an architecture-selection question.

This matrix complements the quantitative values stored in CSV/JSON.

Legend:

- ✓ primary strength
- ○ adequate
- • secondary capability
- × bottleneck


In [ ]:
architectures = ["Single resonator", "Coupled resonators", "Programmable mesh", "Hybrid chip"]
specs = ["Loss", "Squeezing", "Mode control", "Detection", "Feed-forward", "Stability"]

decision = [
    ["✓", "○", "•", "○", "×", "✓"],
    ["○", "✓", "○", "○", "×", "○"],
    ["•", "○", "✓", "○", "○", "•"],
    ["○", "✓", "✓", "✓", "✓", "○"],
]

quantitative = pd.DataFrame(
    [
        [0.75, 0.55, 0.35, 0.55, 0.25, 0.70],
        [0.62, 0.70, 0.55, 0.50, 0.30, 0.58],
        [0.45, 0.65, 0.90, 0.62, 0.55, 0.45],
        [0.55, 0.70, 0.78, 0.75, 0.82, 0.60],
    ],
    index=architectures,
    columns=specs,
)

decision_df = pd.DataFrame(decision, index=architectures, columns=specs)

fig, ax = plt.subplots(figsize=(10.5, 3.8))
ax.axis("off")

table = ax.table(
    cellText=decision_df.reset_index().values,
    colLabels=["Architecture"] + specs,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.08, 1.70)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Architecture Decision Matrix: Which Regime Does Each Design Preserve?", fontsize=15, pad=20)
ax.text(0.5, -0.08, "✓ primary strength   ○ adequate   • secondary capability   × bottleneck",
        ha="center", transform=ax.transAxes, fontsize=10)

quantitative.to_csv(CSV / "37_architecture_regime_scores.csv")
quantitative.to_json(JS / "37_architecture_regime_scores.json", orient="index", indent=2)
decision_df.to_csv(CSV / "37_architecture_decision_matrix.csv")
decision_df.to_json(JS / "37_architecture_decision_matrix.json", orient="index", indent=2)

savefig(fig, "37_07_architecture_decision_matrix.png")
plt.show()
decision_df


## 8. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available. For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "37_outputs.zip"

files_to_zip = [
    FIG / "37_01_operating_window.png",
    FIG / "37_02_constraint_intersection.png",
    FIG / "37_03_active_specifications.png",
    FIG / "37_04_design_progression.png",
    FIG / "37_05_failure_mode_contour.png",
    FIG / "37_06_summary_table.png",
    FIG / "37_07_architecture_decision_matrix.png",
    CSV / "37_active_specifications.csv",
    CSV / "37_summary.csv",
    CSV / "37_architecture_regime_scores.csv",
    CSV / "37_architecture_decision_matrix.csv",
    JS / "37_active_specifications.json",
    JS / "37_summary.json",
    JS / "37_architecture_regime_scores.json",
    JS / "37_architecture_decision_matrix.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- Resources become useful only inside a connected admissible region.
- Operating regimes are intersections of active specifications.
- Engineering selects and preserves the largest connected region of simultaneous admissibility.
- Fault-tolerant computation exists only inside that connected admissible region.
- Robustness asks how much drift the region can tolerate before it fragments.

## Next notebook

**43 — Robust Operating Regions**

Notebook 43 should ask:

```text
How much can the system drift before the admitted operating region fails?
```

That naturally extends Notebook 37 from operating-region identification to robustness.
